[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nekrut/bda/blob/colab/lectures/lecture9.ipynb)

# Lecture 9: Data Manipulation with Pandas

> This is an aggregated tutorial relying on material from:
> - [Justin Bois](http://justinbois.github.io/bootcamp/2020/index.html)
> - [BIOS821 course at Duke](https://people.duke.edu/~ccc14/bios-821-2017/index.html)
> - [Pandas documentation](https://pandas.pydata.org/docs/user_guide/index.html/)

Pandas (from "Panel Data") is an essential piece of scientific (and not only) data analysis infrastructure. It is, in essence, a highly optimized library for manipulating very large tables (or "Data Frames").

## Pandas learning resources

- [Getting started](https://pandas.pydata.org/docs/getting_started/index.html#getting-started) - official introduction from Pandas.
- [Data Science Tools](http://people.duke.edu/~ccc14/bios-821-2017/index.html) - Data Science for Biologists from Duke University.
- [Data Carpentry](https://datacarpentry.org/) - a collection of lessons *à la* Software Carpentry.

In [30]:
# Pandas, conventionally imported as pd
import pandas as pd

Throughout your research career, you will undoubtedly need to handle data, possibly lots of data. The data comes in lots of formats, and you will spend much of your time **wrangling** the data to get it into a usable form.

Pandas is the primary tool in the Python ecosystem for handling data. Its primary object, the `DataFrame` is extremely useful in wrangling data.

# Basics

## The data set

The dataset we will be using is a subset of metadata describing SARS-CoV-2 datasets from the [Sequence Read Archive](https://www.ncbi.nlm.nih.gov/sra).

It is obtained by going to https://www.ncbi.nlm.nih.gov/sra and performing a query with the following search terms: `txid2697049[Organism:noexp]`.

In [31]:
!curl -sLO https://zenodo.org/records/10680001/files/sra_ncov.csv.gz

In [32]:
!gunzip -c sra_ncov.csv.gz | head -n 3

Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
ERR5063394,2021-01-18 12:14:33,0,ERX4869505,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886
ERR5063392,2021-01-18 12:14:33,0,ERX4869504,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,Illumina MiSeq,ERP121228,PRJEB37886
gunzip: error writing to output: Broken pipe
gunzip: sra_ncov.csv.gz: uncompress failed


## Reading in data

Pandas has a very powerful function, `pd.read_csv()` that can read in a CSV file and store the contents in a convenient data structure called a **data frame**.

In [33]:
df = pd.read_csv('sra_ncov.csv.gz')

In [34]:
# View the first few rows
df.head()

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
0,ERR5063394,2021-01-18 12:14:33,0,ERX4869505,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886
1,ERR5063392,2021-01-18 12:14:33,0,ERX4869504,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,Illumina MiSeq,ERP121228,PRJEB37886
2,ERR5063395,2021-01-18 12:14:33,0,ERX4869506,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886
3,ERR5063397,2021-01-18 12:14:33,0,ERX4869510,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886
4,ERR5063396,2021-01-18 12:14:33,0,ERX4869507,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886


## Indexing data frames

The data frame is a convenient data structure for many reasons. Let's start by looking at how data frames are indexed.

**Important**: We index DataFrames by columns, not rows!

In [35]:
# This gives us a column
df['Run'].head()

0    ERR5063394
1    ERR5063392
2    ERR5063395
3    ERR5063397
4    ERR5063396
Name: Run, dtype: object

In [36]:
# Access a single value
df['Run'][4]

'ERR5063396'

However, it's better to use `.loc` for accessing data. This gives the location in the data frame we want.

> **💡 `loc` versus `iloc`:** `loc` uses label-based indexing (actual row and column labels), while `iloc` uses integer-based indexing (integer positions).

In [37]:
df.loc[4, 'Run']

'ERR5063396'

In [38]:
df.iloc[4:6]

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
4,ERR5063396,2021-01-18 12:14:33,0,ERX4869507,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886
5,ERR5063399,2021-01-18 12:14:33,0,ERX4869508,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886


In [39]:
df.iloc[4:6, [0, 2, 4]]

,Run,size_MB,LibraryStrategy
4,ERR5063396,0,AMPLICON
5,ERR5063399,0,Targeted-Capture


In [40]:
df.loc[4:6, ['Run', 'size_MB', 'LibraryStrategy']]

,Run,size_MB,LibraryStrategy
4,ERR5063396,0,AMPLICON
5,ERR5063399,0,Targeted-Capture
6,ERR5063400,0,AMPLICON


## Filtering: Boolean indexing of data frames

Let's say I wanted to pull out accession numbers of runs produced by Pacific Biosciences machines (labeled as `PACBIO_SMRT`). I can use Boolean indexing to specify the row.

In [41]:
df.loc[df['Platform'] == 'PACBIO_SMRT', 'Run']

49441     SRR13144531
49442     SRR13144533
49443     SRR13144530
49444     SRR13144529
49445     SRR13144528
49446     SRR13144534
49447     SRR13144527
49448     SRR13144526
49449     SRR13144525
49450     SRR13144524
49451     SRR13144523
49452     SRR13144532
173270    SRR12038589
173271    SRR12038590
Name: Run, dtype: object

In [42]:
# Pull the whole record
df.loc[df['Platform'] == 'PACBIO_SMRT', :].head(10)

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
49441,SRR13144531,2020-11-25 21:54:15,1137,SRX9584893,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49442,SRR13144533,2020-11-25 21:54:15,1999,SRX9584891,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49443,SRR13144530,2020-11-25 21:54:15,81,SRX9584894,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49444,SRR13144529,2020-11-25 21:54:15,1311,SRX9584895,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49445,SRR13144528,2020-11-25 21:54:15,327,SRX9584896,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49446,SRR13144534,2020-11-25 21:54:15,2292,SRX9584890,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49447,SRR13144527,2020-11-25 21:54:15,3377,SRX9584897,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49448,SRR13144526,2020-11-25 21:54:15,285,SRX9584898,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49449,SRR13144525,2020-11-25 21:54:15,406,SRX9584899,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49450,SRR13144524,2020-11-25 21:54:15,1507,SRX9584900,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710


Now, let's pull out all PacBio records that were obtained from Amplicon sequencing. We can use the `&` operator:

In [43]:
df.loc[(df['Platform'] == 'PACBIO_SMRT') & (df['LibraryStrategy'] == 'AMPLICON'), :].head(3)

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
49441,SRR13144531,2020-11-25 21:54:15,1137,SRX9584893,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49442,SRR13144533,2020-11-25 21:54:15,1999,SRX9584891,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49443,SRR13144530,2020-11-25 21:54:15,81,SRX9584894,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710


In [44]:
# See how many match
import numpy as np
inds = (df['Platform'] == 'PACBIO_SMRT') & (df['LibraryStrategy'] == 'AMPLICON')
np.unique(inds, return_counts=True)

(array([False,  True]), array([190344,     12]))

## Calculating with data frames

Let's add a column that specifies whether or not the corresponding run is above 100 MB:

In [45]:
# Add the column to the DataFrame
df['Over100Mb'] = df['size_MB'] >= 100

# Take a look
df.head()

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject,Over100Mb
0,ERR5063394,2021-01-18 12:14:33,0,ERX4869505,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886,False
1,ERR5063392,2021-01-18 12:14:33,0,ERX4869504,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,Illumina MiSeq,ERP121228,PRJEB37886,False
2,ERR5063395,2021-01-18 12:14:33,0,ERX4869506,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886,False
3,ERR5063397,2021-01-18 12:14:33,0,ERX4869510,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886,False
4,ERR5063396,2021-01-18 12:14:33,0,ERX4869507,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886,False


## A note about vectorization

Notice how applying the `>=` operator to a `Series` resulted in **elementwise** application. This is called **vectorization**. It means that we do not have to write a `for` loop to do operations on the elements of a `Series`.

Vectorized code is almost always faster because the looping is done with compiled code under the hood.

## Outputting a new CSV file

In [46]:
df.to_csv('over100Mb_data.csv', index=False)

In [47]:
!head -3 over100Mb_data.csv

Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject,Over100Mb
ERR5063394,2021-01-18 12:14:33,0,ERX4869505,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886,False
ERR5063392,2021-01-18 12:14:33,0,ERX4869504,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,Illumina MiSeq,ERP121228,PRJEB37886,False


---

# Tidy data

[Hadley Wickham](https://en.wikipedia.org/wiki/Hadley_Wickham) wrote a [great article](http://dx.doi.org/10.18637/jss.v059.i10) in favor of "tidy data." Tidy data frames follow the rules:

1. Each variable is a column.
2. Each observation is a row.
3. Each type of observation has its own separate data frame.

A tidy data frame is almost always **much** easier to work with than non-tidy formats.

## Finding unique values and counts

In [48]:
# Re-read the data (using local file downloaded earlier)
df = pd.read_csv('sra_ncov.csv.gz')
df = df[df['size_MB'] > 0].reset_index(drop=True)

In [49]:
df['Platform'].unique()

array(['ILLUMINA', 'OXFORD_NANOPORE', 'ION_TORRENT', 'PACBIO_SMRT',
       'BGISEQ'], dtype=object)

In [50]:
df['Platform'].value_counts()

Platform
ILLUMINA           155937
OXFORD_NANOPORE     25202
ION_TORRENT           507
BGISEQ                 22
PACBIO_SMRT            14
Name: count, dtype: int64

## Sorting

In [51]:
df_subset = df.sample(n=10)
df_subset

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
50149,ERR4861209,2020-11-21 16:41:51,78,ERX4730592,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
82213,SRR12894498,2020-10-26 02:10:41,33,SRX9359447,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 500,SRP253798,PRJNA613958
72388,ERR4787576,2020-11-04 09:00:36,51,ERX4657334,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
179370,ERR4080522,2020-04-30 11:59:40,48,ERX4077924,WGA,PCR,VIRAL RNA,SINGLE,OXFORD_NANOPORE,MinION,ERP121327,PRJEB37966
31670,ERR4906236,2020-12-04 08:46:14,116,ERX4773060,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
125322,ERR4597347,2020-09-17 10:06:00,80,ERX4531009,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
92941,ERR4686387,2020-10-16 15:55:41,17,ERX4607689,AMPLICON,unspecified,GENOMIC,SINGLE,OXFORD_NANOPORE,MinION,ERP123896,PRJEB40277
80210,SRR12901710,2020-10-26 19:29:02,26,SRX9366412,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina MiSeq,SRP253926,PRJNA614995
45364,ERR4868293,2020-11-27 12:12:51,19,ERX4737733,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 500,ERP121228,PRJEB37886
5201,ERR5055563,2021-01-11 01:05:32,30,ERX4861632,AMPLICON,unspecified,GENOMIC,SINGLE,OXFORD_NANOPORE,GridION,ERP123896,PRJEB40277


In [52]:
df_subset.sort_index()

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
5201,ERR5055563,2021-01-11 01:05:32,30,ERX4861632,AMPLICON,unspecified,GENOMIC,SINGLE,OXFORD_NANOPORE,GridION,ERP123896,PRJEB40277
31670,ERR4906236,2020-12-04 08:46:14,116,ERX4773060,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
45364,ERR4868293,2020-11-27 12:12:51,19,ERX4737733,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 500,ERP121228,PRJEB37886
50149,ERR4861209,2020-11-21 16:41:51,78,ERX4730592,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
72388,ERR4787576,2020-11-04 09:00:36,51,ERX4657334,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
80210,SRR12901710,2020-10-26 19:29:02,26,SRX9366412,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina MiSeq,SRP253926,PRJNA614995
82213,SRR12894498,2020-10-26 02:10:41,33,SRX9359447,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 500,SRP253798,PRJNA613958
92941,ERR4686387,2020-10-16 15:55:41,17,ERX4607689,AMPLICON,unspecified,GENOMIC,SINGLE,OXFORD_NANOPORE,MinION,ERP123896,PRJEB40277
125322,ERR4597347,2020-09-17 10:06:00,80,ERX4531009,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
179370,ERR4080522,2020-04-30 11:59:40,48,ERX4077924,WGA,PCR,VIRAL RNA,SINGLE,OXFORD_NANOPORE,MinION,ERP121327,PRJEB37966


In [53]:
df_subset.sort_values(by=['LibraryLayout', 'size_MB'])

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
45364,ERR4868293,2020-11-27 12:12:51,19,ERX4737733,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 500,ERP121228,PRJEB37886
80210,SRR12901710,2020-10-26 19:29:02,26,SRX9366412,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina MiSeq,SRP253926,PRJNA614995
82213,SRR12894498,2020-10-26 02:10:41,33,SRX9359447,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 500,SRP253798,PRJNA613958
72388,ERR4787576,2020-11-04 09:00:36,51,ERX4657334,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
50149,ERR4861209,2020-11-21 16:41:51,78,ERX4730592,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
125322,ERR4597347,2020-09-17 10:06:00,80,ERX4531009,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
31670,ERR4906236,2020-12-04 08:46:14,116,ERX4773060,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
92941,ERR4686387,2020-10-16 15:55:41,17,ERX4607689,AMPLICON,unspecified,GENOMIC,SINGLE,OXFORD_NANOPORE,MinION,ERP123896,PRJEB40277
5201,ERR5055563,2021-01-11 01:05:32,30,ERX4861632,AMPLICON,unspecified,GENOMIC,SINGLE,OXFORD_NANOPORE,GridION,ERP123896,PRJEB40277
179370,ERR4080522,2020-04-30 11:59:40,48,ERX4077924,WGA,PCR,VIRAL RNA,SINGLE,OXFORD_NANOPORE,MinION,ERP121327,PRJEB37966


In [54]:
df_subset.sort_values(by=['LibraryLayout', 'size_MB'], ascending=[True, False])

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
31670,ERR4906236,2020-12-04 08:46:14,116,ERX4773060,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
125322,ERR4597347,2020-09-17 10:06:00,80,ERX4531009,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
50149,ERR4861209,2020-11-21 16:41:51,78,ERX4730592,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
72388,ERR4787576,2020-11-04 09:00:36,51,ERX4657334,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
82213,SRR12894498,2020-10-26 02:10:41,33,SRX9359447,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 500,SRP253798,PRJNA613958
80210,SRR12901710,2020-10-26 19:29:02,26,SRX9366412,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina MiSeq,SRP253926,PRJNA614995
45364,ERR4868293,2020-11-27 12:12:51,19,ERX4737733,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 500,ERP121228,PRJEB37886
179370,ERR4080522,2020-04-30 11:59:40,48,ERX4077924,WGA,PCR,VIRAL RNA,SINGLE,OXFORD_NANOPORE,MinION,ERP121327,PRJEB37966
5201,ERR5055563,2021-01-11 01:05:32,30,ERX4861632,AMPLICON,unspecified,GENOMIC,SINGLE,OXFORD_NANOPORE,GridION,ERP123896,PRJEB40277
92941,ERR4686387,2020-10-16 15:55:41,17,ERX4607689,AMPLICON,unspecified,GENOMIC,SINGLE,OXFORD_NANOPORE,MinION,ERP123896,PRJEB40277


---

# Split-apply-combine

Let's say we want to compute the total size of SRA runs for each `BioProject`. The strategy is:

1. **Split** the data set up according to the `'BioProject'` field
2. **Apply** a sum function to the split data sets
3. **Combine** the results into a new summary data set

This is the **split-apply-combine** strategy, put forward by Hadley Wickham in [this paper](http://dx.doi.org/10.18637/jss.v040.i01).

## Aggregation

In [55]:
grouped = df.groupby(['BioProject'])
grouped

In [56]:
df_sum = pd.DataFrame(grouped['size_MB'].sum())
df_sum.head(10)

,size_MB
BioProject,
PRJEB37513,19806
PRJEB37886,9235309
PRJEB37966,92058
PRJEB38101,533
PRJEB38351,571
PRJEB38369,1544
PRJEB38388,51684
PRJEB38459,3208
PRJEB38546,1686


In [57]:
df_sum = df_sum.reset_index()
df_sum.head()

,BioProject,size_MB
0,PRJEB37513,19806
1,PRJEB37886,9235309
2,PRJEB37966,92058
3,PRJEB38101,533
4,PRJEB38351,571


In [58]:
# Multiple columns in groupby
df.groupby(['BioProject', 'Platform']).sum(numeric_only=True).reset_index().head(10)

,BioProject,Platform,size_MB
0,PRJEB37513,ILLUMINA,19806
1,PRJEB37886,ILLUMINA,7640033
2,PRJEB37886,OXFORD_NANOPORE,1595276
3,PRJEB37966,OXFORD_NANOPORE,92058
4,PRJEB38101,ILLUMINA,533
5,PRJEB38351,ILLUMINA,571
6,PRJEB38369,ILLUMINA,1544
7,PRJEB38388,OXFORD_NANOPORE,51684
8,PRJEB38459,ILLUMINA,2672
9,PRJEB38459,OXFORD_NANOPORE,536


In [59]:
# Descriptive statistics
df.groupby(['BioProject', 'Platform'])['size_MB'].describe().head(10)

count         mean         std     min  \
BioProject Platform                                                     
PRJEB37513 ILLUMINA            244.0    81.172131   35.651535    21.0   
PRJEB37886 ILLUMINA         114178.0    66.913355   67.270998     1.0   
           OXFORD_NANOPORE   19128.0    83.400042  115.955997     1.0   
PRJEB37966 OXFORD_NANOPORE     752.0   122.417553  168.301885     5.0   
PRJEB38101 ILLUMINA              6.0    88.833333   75.058422     4.0   
PRJEB38351 ILLUMINA              6.0    95.166667   10.496031    80.0   
PRJEB38369 ILLUMINA             18.0    85.777778   24.673290    49.0   
PRJEB38388 OXFORD_NANOPORE     583.0    88.651801   65.441487     1.0   
PRJEB38459 ILLUMINA              1.0  2672.000000         NaN  2672.0   
           OXFORD_NANOPORE       1.0   536.000000         NaN   536.0   

                                25%     50%      75%     max  
BioProject Platform                                           
PRJEB37513 ILLUMINA           36.00    92.5   108.00   161.0  
PRJEB37886 ILLUMINA           22.00    59.0    89.00  3536.0  
           OXFORD_NANOPORE    19.00    51.0   108.00  3845.0  
PRJEB37966 OXFORD_NANOPORE    67.00   110.0   150.00  2846.0  
PRJEB38101 ILLUMINA           25.00    89.0   147.00   181.0  
PRJEB38351 ILLUMINA           88.75    96.0   103.25   107.0  
PRJEB38369 ILLUMINA           61.25    92.5   104.00   124.0  
PRJEB38388 OXFORD_NANOPORE    44.00    75.0   122.00   421.0  
PRJEB38459 ILLUMINA         2672.00  2672.0  2672.00  2672.0  
           OXFORD_NANOPORE   536.00   536.0   536.00   536.0

In [60]:
# Custom aggregations
df.groupby(['BioProject', 'Platform']).agg({'size_MB': np.mean, 'Run': 'nunique'}).head(10)

/var/folders/f0/hrhg01rs28sgr9r5v8k3rwrw0000gn/T/ipykernel_69872/3276325559.py:2: FutureWarning: The provided callable <function mean at 0x1103fe5c0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby(['BioProject', 'Platform']).agg({'size_MB': np.mean, 'Run': 'nunique'}).head(10)


size_MB     Run
BioProject Platform                            
PRJEB37513 ILLUMINA           81.172131     244
PRJEB37886 ILLUMINA           66.913355  114178
           OXFORD_NANOPORE    83.400042   19128
PRJEB37966 OXFORD_NANOPORE   122.417553     752
PRJEB38101 ILLUMINA           88.833333       6
PRJEB38351 ILLUMINA           95.166667       6
PRJEB38369 ILLUMINA           85.777778      18
PRJEB38388 OXFORD_NANOPORE    88.651801     583
PRJEB38459 ILLUMINA         2672.000000       1
           OXFORD_NANOPORE   536.000000       1

---

## Tidying a data set with melt

The most useful function for tidying data is `pd.melt()`. Let's demonstrate with a coverage dataset:

In [61]:
df_cov = pd.read_csv('https://zenodo.org/records/10680470/files/coverage.tsv.gz', sep='\t')
df_cov.head()

,#chr,start,end,SRR12733539.bam,SRR12733570.bam,SRR12733581.bam,SRR12733607.bam,SRR12733616.bam,SRR12733619.bam
0,NC_045512.2,0,100,259.0,3490.0,1645.0,399.0,598.0,107.0
1,NC_045512.2,100,200,720.0,9106.0,3780.0,1063.0,1404.0,272.0
2,NC_045512.2,200,300,979.0,14207.0,5054.0,1450.0,1921.0,344.0
3,NC_045512.2,300,400,983.0,15438.0,5212.0,1396.0,1908.0,328.0
4,NC_045512.2,400,500,1297.0,17257.0,6174.0,1567.0,2021.0,385.0


These data are not tidy. When we melt the data frame, the data within it (called **values**) become a single column. The headers (called **variables**) also become new columns.

![Pandas melt operation diagram showing wide format table transformed to long format with id_vars, variable, and value columns](https://pandas.pydata.org/docs/_images/07_melt.svg)

In [62]:
melted = pd.melt(df_cov, 
                 value_name='coverage', 
                 var_name='sample',
                 value_vars=df_cov.columns[3:],
                 id_vars=['start', 'end'])

melted.head()

,start,end,sample,coverage
0,0,100,SRR12733539.bam,259.0
1,100,200,SRR12733539.bam,720.0
2,200,300,SRR12733539.bam,979.0
3,300,400,SRR12733539.bam,983.0
4,400,500,SRR12733539.bam,1297.0


In [63]:
melted.groupby(['sample'])['coverage'].describe()

,count,mean,std,min,25%,50%,75%,max
sample,,,,,,,,
SRR12733539.bam,100.0,1271.75,295.525499,259.0,1093.00,1310.0,1484.25,1963.0
SRR12733570.bam,100.0,10112.04,3412.618417,1843.0,8582.25,10615.5,12062.75,18673.0
SRR12733581.bam,100.0,5800.16,1360.497731,1645.0,5038.50,6118.5,6766.50,8100.0
SRR12733607.bam,100.0,1509.15,389.486359,399.0,1334.50,1571.0,1834.75,2159.0
SRR12733616.bam,100.0,1691.37,444.477395,414.0,1446.50,1820.0,2030.00,2366.0
SRR12733619.bam,100.0,359.54,77.393318,107.0,321.50,384.5,411.00,482.0


To get back from melted (narrow) format to wide format, use `pivot()`:

![Pandas pivot operation diagram showing long format table transformed back to wide format](https://pandas.pydata.org/docs/_images/07_pivot.svg)

In [64]:
melted.pivot(index=['start', 'end'], columns='sample', values='coverage').head()

,sample,SRR12733539.bam,SRR12733570.bam,SRR12733581.bam,SRR12733607.bam,SRR12733616.bam,SRR12733619.bam
start,end,,,,,,
0,100,259.0,3490.0,1645.0,399.0,598.0,107.0
100,200,720.0,9106.0,3780.0,1063.0,1404.0,272.0
200,300,979.0,14207.0,5054.0,1450.0,1921.0,344.0
300,400,983.0,15438.0,5212.0,1396.0,1908.0,328.0
400,500,1297.0,17257.0,6174.0,1567.0,2021.0,385.0


---

# Working with multiple tables

Working with multiple tables often involves joining them on a common key.

![Pandas merge left join diagram showing how two dataframes combine on matching keys](https://pandas.pydata.org/docs/_images/08_merge_left.svg)

In [65]:
df1 = pd.DataFrame({"key": ["A", "B", "C", "D"], "value": np.random.randn(4)})
df2 = pd.DataFrame({"key": ["B", "D", "D", "E"], "value": np.random.randn(4)})

In [66]:
df1

,key,value
0,A,2.159150
1,B,0.243231
2,C,0.577176
3,D,0.675909


In [67]:
df2

,key,value
0,B,-0.262216
1,D,0.093675
2,D,-0.408300
3,E,0.069736


## Inner join

![Venn diagram showing inner join - intersection of sets A and B highlighted](https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/SQL_Join_-_07_A_Inner_Join_B.svg/234px-SQL_Join_-_07_A_Inner_Join_B.svg.png)

In [68]:
pd.merge(df1, df2, on="key")

,key,value_x,value_y
0,B,0.243231,-0.262216
1,D,0.675909,0.093675
2,D,0.675909,-0.408300


## Left join

![Venn diagram showing left join - all of set A plus intersection with B highlighted](https://upload.wikimedia.org/wikipedia/commons/thumb/d/dc/SQL_Join_-_01b_A_Left_Join_B.svg/234px-SQL_Join_-_01b_A_Left_Join_B.svg.png)

In [69]:
pd.merge(df1, df2, on="key", how="left").fillna('.')

,key,value_x,value_y
0,A,2.159150,.
1,B,0.243231,-0.262216
2,C,0.577176,.
3,D,0.675909,0.093675
4,D,0.675909,-0.4083


## Right join

![Venn diagram showing right join - all of set B plus intersection with A highlighted](https://upload.wikimedia.org/wikipedia/commons/thumb/5/5f/SQL_Join_-_03_A_Right_Join_B.svg/234px-SQL_Join_-_03_A_Right_Join_B.svg.png)

In [70]:
pd.merge(df1, df2, on="key", how="right").fillna('.')

,key,value_x,value_y
0,B,0.243231,-0.262216
1,D,0.675909,0.093675
2,D,0.675909,-0.408300
3,E,.,0.069736


## Full (outer) join

![Venn diagram showing full outer join - all of both sets A and B highlighted](https://upload.wikimedia.org/wikipedia/commons/thumb/6/61/SQL_Join_-_05_A_Full_Join_B.svg/234px-SQL_Join_-_05_A_Full_Join_B.svg.png)

In [71]:
pd.merge(df1, df2, on="key", how="outer").fillna('.')

,key,value_x,value_y
0,A,2.15915,.
1,B,0.243231,-0.262216
2,C,0.577176,.
3,D,0.675909,0.093675
4,D,0.675909,-0.4083
5,E,.,0.069736


---

# Putting it all together: Pandas + Altair

## Understanding [Altair](https://altair-viz.github.io/)

Vega-Altair is a declarative statistical visualization library for Python. It offers a powerful and concise grammar that enables you to quickly build a wide range of statistical visualizations.

In [72]:
import pandas as pd
import altair as alt
from datetime import date
today = date.today()

In [73]:
# Read a larger dataset
sra = pd.read_csv(
    "https://zenodo.org/records/10680776/files/ena.tsv.gz",
    compression='gzip',
    sep="\t",
    low_memory=False,
    nrows=100000  # Limit rows for faster loading
)

In [74]:
len(sra)

100000

In [75]:
sra.sample(5)

,study_accession,base_count,accession,collection_date,country,culture_collection,description,sample_collection,sample_title,sequencing_method,...,library_name,library_construction_protocol,library_layout,instrument_model,instrument_platform,isolation_source,isolate,investigation_type,collection_date_submitted,center_name
31562,PRJNA716984,4477222.0,SAMN21918147,2021-09-07,USA: Indiana,NaN,Sequel II sequencing,NaN,CDC Sars CoV2 Sequencing Baseline Constellation,NaN,...,Unknown,Freed primers,PAIRED,Sequel II,PACBIO_SMRT,Nasal Swabs,SARS-CoV-2/Human/USA/IN-CDC-LC0287206/2021,NaN,2021-09-07,NaN
87931,PRJNA856404,33599098.0,SAMN29960563,2022-05-14,Australia: Victoria,NaN,GridION sequencing; Whole genome sequencing of...,NaN,This sample has been submitted by pda|williams...,NaN,...,RATED17,NaN,SINGLE,GridION,OXFORD_NANOPORE,Rapid antigen test device,RATED17,NaN,2022-05-14,SUB11855608
68138,PRJEB37886,842505955.0,SAMEA11047407,2021-11-26,United Kingdom,NaN,Illumina NovaSeq 6000 sequencing; Illumina Nov...,NaN,COG-UK/MILK-2C13DE3,NaN,...,NT1711934H / HT-125130:A7,NaN,PAIRED,Illumina NovaSeq 6000,ILLUMINA,NaN,not provided,NaN,2021-11-26,SC
22123,PRJNA625551,39200317.0,SAMN23710884,2021-01-01,USA: Virginia,NaN,MinION sequencing; Amplicon-based sequencing o...,NaN,Amplicon-based sequencing of SARS-CoV-2 : VA-C...,NaN,...,SARS-CoV-2 : VA-CAV_VAS3N_00004000_01,NaN,SINGLE,MinION,OXFORD_NANOPORE,human,NaN,NaN,2021,SUB10769604
17979,PRJNA686984,38069750.0,SAMN25038967,2021-12-21,USA: Colorado,NaN,NextSeq 550 sequencing; PCR tiled amplicon WGS...,NaN,PCR tiled amplicon WGS of SARS-CoV-2,NaN,...,112,NaN,SINGLE,NextSeq 550,ILLUMINA,patient isolate,CO-CDPHE-2102510306,NaN,2021-12-21,SUB10962253


## Cleaning the data

In [76]:
# Convert collection_date to datetime
# Note: errors='coerce' converts unparseable dates to NaT (Not a Time)
# This is acceptable here because we will filter out invalid dates in the next step
sra = sra.assign(collection_date=pd.to_datetime(sra["collection_date"], errors='coerce'))

In [77]:
print('Earliest entry:', sra['collection_date'].min())
print('Latest entry:', sra['collection_date'].max())

Earliest entry: 2019-12-30 00:00:00
Latest entry: 2023-01-20 00:00:00


> **⚠️ Data Quality:** Don't get surprised here - the metadata is only as good as the person who entered it. So, **when you enter metadata for your sequencing data -- pay attention!!!**

In [78]:
# Filter to valid date range using explicit Timestamp objects for clarity
sra = sra[
    (sra['collection_date'] >= pd.Timestamp('2020-01-01')) 
    & 
    (sra['collection_date'] <= pd.Timestamp('2023-12-31'))
]

In [79]:
# Aggregate data for heatmap
heatmap_2d = sra.groupby(
    ['instrument_platform', 'library_strategy']
).agg(
    {'run_accession': 'nunique'}
).reset_index()

heatmap_2d

,instrument_platform,library_strategy,run_accession
0,BGISEQ,AMPLICON,1
1,BGISEQ,OTHER,13
2,BGISEQ,RNA-Seq,2
3,BGISEQ,Targeted-Capture,2
4,DNBSEQ,AMPLICON,3
5,ILLUMINA,AMPLICON,78262
6,ILLUMINA,OTHER,2
7,ILLUMINA,RNA-Seq,524
8,ILLUMINA,Targeted-Capture,272
9,ILLUMINA,WCS,2


## Plotting the data

In [80]:
back = alt.Chart(heatmap_2d).mark_rect(opacity=1).encode(
    x=alt.X(
        "instrument_platform:N",
        title="Instrument"
    ),
    y=alt.Y(
        "library_strategy:N",
        title="Strategy",
        axis=alt.Axis(orient='right')
    ),
    color=alt.Color(
        "run_accession:Q",
        title="# Samples",
        scale=alt.Scale(
            scheme="goldred",
            type="log"
        ),
    ),
    tooltip=[
        alt.Tooltip(
            "instrument_platform:N",
            title="Machine"
        ),
        alt.Tooltip(
            "run_accession:Q",
            title="Number of runs"
        ),
        alt.Tooltip(
            "library_strategy:N",
            title="Protocol"
        )
    ]
).properties(
    width=500,
    height=150,
    title={
        "text": ["Breakdown of datasets from ENA",
                 "by Platform and Library Strategy"],
        "subtitle": "(Sample of 100k records)"
    }
)

back

alt.Chart(...)

In [81]:
# Add text labels
front = back.mark_text(
    align="center",
    baseline="middle",
    fontSize=12,
    fontWeight="bold",
).encode(
    text=alt.Text("run_accession:Q", format=",.0f"),
    color=alt.condition(
        alt.datum.run_accession > 200,
        alt.value("white"),
        alt.value("black")
    )
)

# Combine layers
back + front

alt.LayerChart(...)

## Summary

In this lecture, we covered:

1. **Pandas basics**: DataFrames, indexing with `loc` and `iloc`
2. **Boolean indexing**: Filtering data with conditions
3. **Calculations**: Vectorized operations on columns
4. **Tidy data**: Principles of data organization
5. **Split-apply-combine**: Using `groupby()` for aggregations
6. **Reshaping**: `melt()` and `pivot()` for transforming data
7. **Joins**: Combining tables with `merge()`
8. **Visualization**: Creating plots with Altair

These skills form the foundation of data analysis in Python!